In [2]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import speech_recognition as sr
from pydub import AudioSegment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from wordcloud import WordCloud
from collections import Counter
import re
import unicodedata
from pyproj import Transformer
import pyproj

c:\Users\Jonny Villareal\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [3]:
dia = "20260519"

In [4]:
#Importar archivos desglosados

md = pd.read_csv(f'Z:/01 base_datos/06 matriz_distancia_FMS/{dia}_matriz distancias.csv', encoding='utf-8-sig')
    
md.head(2)

,Tipo de Servicio,Id Línea,Línea,Configuraciones,Configuración Activa,Id Sublínea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"Sinóptico, Horario GOAL, Punto de control (TM-..."
1,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de Engativá Emaús,138.0,595258.0,521149.0,NaN


In [5]:
# Convertir a numérico (por seguridad)
md['Coordenada X'] = pd.to_numeric(md['Coordenada X'], errors='coerce')
md['Coordenada Y'] = pd.to_numeric(md['Coordenada Y'], errors='coerce')

# Rellenar NaN con 0 y convertir a int
md['Coordenada X'] = md['Coordenada X'].fillna(0).astype(int)
md['Coordenada Y'] = md['Coordenada Y'].fillna(0).astype(int)

In [6]:
from pyproj import Transformer
import pandas as pd
import pyproj

# Definir las proyecciones UTM y latitud/longitud
utm_projection = pyproj.Proj(proj='utm', zone=17, ellps='WGS84', north=True)
lat_lon_projection = pyproj.Proj(proj='latlong', ellps='WGS84')

# Función para convertir coordenadas UTM a latitud y longitud
def convertir_a_lat_lon(x_utm, y_utm):
    longitud, latitud = pyproj.transform(utm_projection, lat_lon_projection, x_utm, y_utm)
    return latitud, longitud

# Aplicar la función a las columnas x_prevloc, y_prevloc, x_loc, y_loc
md[['latitud', 'longitud']] = md.apply(lambda row: pd.Series(convertir_a_lat_lon(row['Coordenada X'], row['Coordenada Y'])), axis=1)

md.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_44428\824314655.py:11: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  longitud, latitud = pyproj.transform(utm_projection, lat_lon_projection, x_utm, y_utm)


,Tipo de Servicio,Id Línea,Línea,Configuraciones,Configuración Activa,Id Sublínea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,latitud,longitud
0,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348,521193,"Sinóptico, Horario GOAL, Punto de control (TM-...",4.714758,-80.140274
1,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de Engativá Emaús,138.0,595258,521149,NaN,4.714361,-80.141086
2,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52365.0,068A05_TM,068A05_Liceo Salomón Sabio,425.0,595014,520997,NaN,4.712989,-80.143287
3,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52442.0,110A05_TM,110A05_Br. Sabana del Dorado,686.0,594933,520819,NaN,4.711379,-80.144020
4,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52369.0,070A05_TM,070A05_Br. Sabana del Dorado,964.0,595091,520593,NaN,4.709333,-80.142598


In [9]:
md["cenefa"] = md["Etiqueta Nodo"].str.split("_").str[0]

md["cenefa"] = md["cenefa"].astype(str).str.strip().str.upper()

md.head()

,Tipo de Servicio,Id Línea,Línea,Configuraciones,Configuración Activa,Id Sublínea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,latitud,longitud,cenefa
0,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348,521193,"Sinóptico, Horario GOAL, Punto de control (TM-...",4.714758,-80.140274,247A05
1,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de Engativá Emaús,138.0,595258,521149,NaN,4.714361,-80.141086,222A05
2,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52365.0,068A05_TM,068A05_Liceo Salomón Sabio,425.0,595014,520997,NaN,4.712989,-80.143287,068A05
3,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52442.0,110A05_TM,110A05_Br. Sabana del Dorado,686.0,594933,520819,NaN,4.711379,-80.144020,110A05
4,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52369.0,070A05_TM,070A05_Br. Sabana del Dorado,964.0,595091,520593,NaN,4.709333,-80.142598,070A05


In [8]:
ruta_exportacion = (
    f'C:/Users/Jonny Villareal/'
    f'OneDrive - Gmovil SAS/'
    f'Escritorio/Jonny/'
    f'Control 2026/'
    f'Automatización/'
    f'{dia}_md coord.csv'
)

md.to_csv(
    ruta_exportacion,
    sep=';',
    index=False,
    encoding='utf-8-sig'
)